In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error



In [2]:
# 1. Load Data
df = pd.read_csv("Delivery_Logistics.csv")

In [3]:
# 2. Drop ID and LEAKY features (Columns known only AFTER delivery)
# 'delivery_id' is useless for prediction.
# 'delivery_status', 'delayed', 'delivery_rating', 'actual_time' are future info.
cols_to_drop = [
    'delivery_id', 
    'delivery_partner',  # Keep this if you want to analyze partner costs, otherwise drop
    'delivery_time_hours', 
    'expected_time_hours', 
    'delivery_status', 
    'delayed', 
    'delivery_rating'   # CRITICAL: Removing leakage
]
df.drop(columns=cols_to_drop, inplace=True)

In [4]:
# 3. Encode Categorical Variables (One-line fix)
# This automatically converts package_type, vehicle_type, etc.
df = pd.get_dummies(df, drop_first=True)

In [5]:
# 4. Split Data
X = df.drop('delivery_cost', axis=1)
y = df['delivery_cost']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
# 5. Scale Data (Crucial for Linear Models)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [7]:

# 6. Train & Evaluate
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(),
    "Gradient Boosting": GradientBoostingRegressor()
}

for name, model in models.items():
    # Use scaled data for linear regression, unscaled is fine for trees but scaled won't hurt
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    r2 = r2_score(y_test, y_pred)
    print(f"{name} R2 Score: {r2:.4f}")

# If your R2 is now around 0.60 - 0.90, that is REALISTIC. 
# If it drops to 0.999 again, check for leakage (e.g., is Cost a simple formula of Distance?)

Linear Regression R2 Score: 0.9999
Random Forest R2 Score: 0.9998
Gradient Boosting R2 Score: 0.9997
